In [2]:

import pandas as pd
import re
from collections import Counter

from utils import split_sentences, split_by_commas, count_words, chunk_text

# **1. PREPARATION DU JEU DE DONNEES**

## **1.1. Import et nottoyage du jeu de données**

In [7]:
# Ouverture du jeu de données et transformation en un Dataframe
df = pd.read_excel("data/Data_memoire.xlsx")
df

,AUTEUR,LANGUE,VILLE,TEXTE
0,AL YAQUBI,arabe,Le Caire,الغسطاط تعرف بباب اليُون وهو الموضع المعروف با...
1,IBN RUSTAH,arabe,Le Caire,ومن عجائب البلدان الهرمان ة بمصر سمك كل واحد م...
2,AL-MASUDI,arabe,Le Caire,والثالثة فسطاط مصر كان تمصير عمرو ابن العاص فس...
3,IBN HAWQAL,arabe,Le Caire,ومن صفات مدنها وبقاعها أنّ مدينتها العُظمى تسم...
4,IBN HAWQAL,arabe,Palerme,ذكر صقلية وأما صقلية فجزيرة طولها سبعة أيام في...
5,IBN HAWQAL,arabe,Cordoue,وأعظم مدينة بالاندلس قُرطبه وليس بجميع المغرب ...
6,AL-MUQADDASI,arabe,Le Caire,الفُنطَاط هو مصر في كلّ قول لأنه قد جمع الدواو...
7,AL-MUQADDASI,arabe,Palerme,بلرم هى قصبة اصقلية ، على البحر فى الجزيرة ، ا...
8,AL-MUQADDASI,arabe,Cordoue,[٢٣٣] قرطبة هى مصر الاندلس ؛ سمعت بعض العثماني...
9,AL-IDRISI,arabe,Le Caire,ومدينة الفسطاط هي مصر سميت بذلك لأن مصرام بن ح...


In [8]:
# Nettoyage du jeu de donnnées des caractères spéciaux
df['TEXTE'] = (
    df['TEXTE']
    .str.lower()                                    # tout en minuscule
    .str.replace(r'\[[^\]]*\]', '', regex=True)     # enlève [nombre] ou [lettre]
    .str.replace(r'\(\d+\)', '', regex=True)        # enlève (nombre)
    .str.replace(r'\s{2,}', ' ', regex=True)        # remplace les doubles espaces par un seul
    .str.replace(r'\n{2,}', '\n', regex=True)       # remplace les sauts de ligne multiples par un seul
    .str.replace(r'[\*°|/]', '', regex=True)        # Supprimer ponctuations inutiles
    .str.replace(r'[\[\]\(\)]', '', regex=True)     # Supprimer crochets et parenthèses orphelins
    .str.replace('\u200e', '', regex=False)         # Supprimer caractère invisible (Left-to-Right mark, etc.)
    .str.replace('§', '', regex=False)              # Supprimer symbole de paragraphe
    .str.replace(r"[‘’]", "'", regex=True)          # Normaliser apostrophes
    .str.replace(r'[«»“”"]', '"', regex=True)       # Normaliser guillemets
    .str.replace('•', '.', regex=False)             # Remplacer puces par un point
    .str.replace(' .', '.', regex=False)             # Remplacer puces par un point
    .str.replace(r'[\u064B-\u0652]', '', regex=True)   # Supprimer les diacritiques arabes (tashkil)
    .str.strip()                                    # supprime les espaces de début/fin
)

In [9]:
# Verificiations des caratères spéciaux restants
all_text = ' '.join(df['TEXTE'].dropna())        # Concatène tous les textes en un seul
special_chars = re.findall(r'[^\w\s]', all_text) # Extrait tous les caractères spéciaux (non lettres, non chiffres, non espace)
special_char_counts = Counter(special_chars)     # Compte la fréquence de chaque caractère spécial

# Affiche les caractères spéciaux uniques triés par fréquence
for char, count in special_char_counts.most_common():
    print(f"'{char}': {count}")

'،': 756
'.': 373
',': 247
'"': 113
':': 82
'-': 59
'؛': 58
'ٰ': 21
';': 16
''': 14
'؟': 8


## **1.2. Découpage en chunks de mots**

In [11]:
# --- Application sur le DataFrame ---
rows = []
for _, row in df.iterrows():
    chunks = chunk_text(row['TEXTE'])
    for c in chunks:
        rows.append({
            'AUTEUR': row['AUTEUR'],
            'LANGUE': row['LANGUE'],
            'VILLE': row['VILLE'],
            'CHUNK_MOTS': c,
            'NB_MOTS': count_words(c)
        })

df_chunks = pd.DataFrame(rows)

In [12]:
df_chunks.head(10)

,AUTEUR,LANGUE,VILLE,CHUNK_MOTS,NB_MOTS
0,AL YAQUBI,arabe,Le Caire,الغسطاط تعرف بباب اليون وهو الموضع المعروف بال...,50
1,AL YAQUBI,arabe,Le Caire,عمرو بن العاص مسجد جامعها ودار امارتها المعروف...,36
2,AL YAQUBI,arabe,Le Caire,واسكنه قوما وكتب الى عمر بن الخطاب بذلك فكتب ا...,50
3,AL YAQUBI,arabe,Le Caire,لان لكل كورة مدينة مخصوصة بأمر من الامور من مد...,56
4,AL YAQUBI,arabe,Le Caire,ومدينة القيس وبها تعل الثياب القيسية والأكسية ...,50
5,AL YAQUBI,arabe,Le Caire,فى لجانب الشرقى من النيل ومدينة الأشمونين وبها...,53
6,AL YAQUBI,arabe,Le Caire,ولهما ساحل وبها يعمل الفرش القطوع والجلود الاخ...,28
7,IBN RUSTAH,arabe,Le Caire,ومن عجائب البلدان الهرمان ة بمصر سمك كل واحد م...,50
8,IBN RUSTAH,arabe,Le Caire,لها الرعادة من مسها وجد خدرا في كفه ويده وذراع...,50
9,IBN RUSTAH,arabe,Le Caire,و فوضع احد طرفيها على هذه الشبكة وامسك الطرف ا...,50


In [13]:
print(f"Nombre de chunks : {len(df_chunks)}")
print(f"Nombre de petits chunks ( - de 10 mots) : {len(df_chunks[df_chunks['NB_MOTS'] < 10])}")  # Vérification des petits chunks
print(f"Nombre de grand chunks (entre 50 et 60 mots) : {len(df_chunks[(df_chunks['NB_MOTS'] > 50) & (df_chunks['NB_MOTS'] < 60)])}")

Nombre de chunks : 609
Nombre de petits chunks ( - de 10 mots) : 0
Nombre de grand chunks (entre 50 et 60 mots) : 74


## **1.3. Numerotation des chunks**

In [14]:
df_chunks['CHUNK_ID'] = df_chunks.groupby(['AUTEUR', 'VILLE']).cumcount() + 1
df_chunks.head(100)

,AUTEUR,LANGUE,VILLE,CHUNK_MOTS,NB_MOTS,CHUNK_ID
0,AL YAQUBI,arabe,Le Caire,الغسطاط تعرف بباب اليون وهو الموضع المعروف بال...,50,1
1,AL YAQUBI,arabe,Le Caire,عمرو بن العاص مسجد جامعها ودار امارتها المعروف...,36,2
2,AL YAQUBI,arabe,Le Caire,واسكنه قوما وكتب الى عمر بن الخطاب بذلك فكتب ا...,50,3
3,AL YAQUBI,arabe,Le Caire,لان لكل كورة مدينة مخصوصة بأمر من الامور من مد...,56,4
4,AL YAQUBI,arabe,Le Caire,ومدينة القيس وبها تعل الثياب القيسية والأكسية ...,50,5
...,...,...,...,...,...,...
95,AL-MUQADDASI,arabe,Le Caire,تنيس بين بحر الروم والنيل بحيرة فيها جزيرة صغي...,50,23
96,AL-MUQADDASI,arabe,Le Caire,والماء في صهاريج مغلقة أكثر أهلها قبط والبلاذا...,30,24
97,AL-MUQADDASI,arabe,Le Caire,دمياط تسير في هذه البحيرة يوما وليلة ربما لقيك...,50,25
98,AL-MUQADDASI,arabe,Le Caire,كثيرة حزبة ولهم موسم كل سنة يقصدها المرابطون م...,25,26


In [15]:
lignes_par_groupe = []
for (auteur, ville), group in df_chunks.groupby(['AUTEUR', 'VILLE']):
    premiere_ligne = group.index.min()
    derniere_ligne = group.index.max()
    lignes_par_groupe.append({'AUTEUR': auteur, 'VILLE': ville, 'Première Ligne': premiere_ligne, 'Dernière Ligne': derniere_ligne})
df_lignes = pd.DataFrame(lignes_par_groupe)

# Afficher le résultat
print("Plage de lignes pour chaque auteur et ville dans df_chunks:")
df_lignes.sort_values(by=["Première Ligne"],ascending=True)


Plage de lignes pour chaque auteur et ville dans df_chunks:


,AUTEUR,VILLE,Première Ligne,Dernière Ligne
1,AL YAQUBI,Le Caire,0,6
15,IBN RUSTAH,Le Caire,7,18
6,AL-MASUDI,Le Caire,19,20
17,IBN HAWQAL,Le Caire,21,30
18,IBN HAWQAL,Palerme,31,56
16,IBN HAWQAL,Cordoue,57,72
8,AL-MUQADDASI,Le Caire,73,137
9,AL-MUQADDASI,Palerme,138,139
7,AL-MUQADDASI,Cordoue,140,142
4,AL-IDRISI,Le Caire,143,176


In [ ]:
# Sauvegarde du dataframe
df_chunks.to_excel('data/df_chunks_part1.xlsx', index=False)